### np.linalg.inv

In [5]:
def get_shape(a):
    if not isinstance(a, list):
        return ()
    if len(a) == 0:
        return (0,)
    inner_shape = get_shape(a[0])
    for i in range(1, len(a)):
        if get_shape(a[i]) != inner_shape:
            raise ValueError(
                f"Jagged array detected: length of {a[i]} not matching {inner_shape}"
            )
    return (len(a),) + inner_shape


def deep_copy(a):
    if isinstance(a, list):
        return [deep_copy(item) for item in a]
    return a


def np_linalg_inv(a):
    # Handle 0x0 matrix before shape validation
    if a == []:
        return []
    
    a = deep_copy(a)
    shape = get_shape(a)

    if len(shape) != 2:
        raise ValueError("function only accepts 2D arrays")
    if shape[0] != shape[1]:
        raise ValueError("Last 2 dimensions of the array must be square")
    if shape == (0, 0):
        return []
    if shape == (1, 1):
        return [[1.0 / a[0][0]]]

    n = shape[0]

    # Build augmented matrix [A | I]
    aug = [
        [float(a[i][j]) for j in range(n)] + [1.0 if i == j else 0.0 for j in range(n)]
        for i in range(n)
    ]

    # Gauss-Jordan elimination with partial pivoting
    for i in range(n):
        # Find pivot row with largest absolute value
        max_row = i
        for k in range(i + 1, n):
            if abs(aug[k][i]) > abs(aug[max_row][i]):
                max_row = k

        # Singular matrix check
        if abs(aug[max_row][i]) < 1e-12:
            raise ValueError("Matrix is singular")

        # Swap current row with pivot row
        aug[i], aug[max_row] = aug[max_row], aug[i]

        # Normalize pivot row
        pivot = aug[i][i]
        for j in range(2 * n):
            aug[i][j] /= pivot

        # Eliminate column i from all other rows
        for k in range(n):
            if k != i:
                factor = aug[k][i]
                for j in range(2 * n):
                    aug[k][j] -= factor * aug[i][j]

    # Extract right half
    return [row[n:] for row in aug]

### test cases 

In [6]:
print(np_linalg_inv([[1, 2], [3, 4]]))  

print(np_linalg_inv([[1, 0, 0], [0, 1, 0], [0, 0, 1]]))  

print(np_linalg_inv([[2, 0], [0, 3]]))  

print(np_linalg_inv([[4]]))  

print(np_linalg_inv([])) 

print(np_linalg_inv([[-5]])) 

print(np_linalg_inv([[1, 2, 3], [0, 1, 4], [5, 6, 0]])) 

[[-1.9999999999999996, 0.9999999999999998], [1.4999999999999998, -0.49999999999999994]]
[[1.0, 0.0, 0.0], [0.0, 1.0, 0.0], [0.0, 0.0, 1.0]]
[[0.5, 0.0], [0.0, 0.3333333333333333]]
[[0.25]]
[]
[[-0.2]]
[[-23.99999999999998, 17.999999999999986, 4.9999999999999964], [19.999999999999982, -14.999999999999988, -3.999999999999997], [-4.999999999999996, 3.999999999999997, 0.9999999999999992]]


In [7]:
print(np_linalg_inv([[1, 2, 3], [4, 5, 6]]))

ValueError: Last 2 dimensions of the array must be square

In [8]:
print(np_linalg_inv([[1, 2], [2, 4]]))

ValueError: Matrix is singular

In [9]:
print(np_linalg_inv([1, 2, 3]))

ValueError: function only accepts 2D arrays

In [10]:
print(np_linalg_inv(42) )

ValueError: function only accepts 2D arrays

In [11]:
print(np_linalg_inv([[1, 2], [3]]))

ValueError: Jagged array detected: length of [3] not matching (2,)

In [12]:
print(np_linalg_inv([[0]]))

ZeroDivisionError: float division by zero